# Step 3: Fix Data Leakage via Chronological Expanding-Window Profiling
**MSc Data Science Thesis - University of Wolverhampton**

###  Methodology: Strict Temporal Isolation
To prevent **Target Data Leakage**, historical customer return rates, average review scores, and product return statistics must be computed **strictly prior** to the current order timestamp ($T_i < T_{\text{current}}$).

###  Mathematical Expanding Window Formulations:
1. **Customer Historical Return Rate**:
   $$\text{customer\_return\_rate}_i = \frac{\sum_{k=1}^{i-1} y_{c, k}}{i - 1} \quad (\text{defaults to } 0.0 \text{ for first order})$$
2. **Product Historical Return Rate**:
   $$\text{product\_return\_rate}_i = \frac{\sum_{k=1}^{i-1} y_{p, k}}{i - 1} \quad (\text{defaults to global mean } 0.1562)$$
3. **Output Checkpoint**: `processed_return_data_no_leakage.csv`.

In [1]:
import pandas as pd
import numpy as np

print("Step 3: Calculating clean, leak-free historical features chronologically...")

# 1. Load data output directly from Step 2 checkpoint
df = pd.read_csv('consolidated_cleaned_data.csv')
print(f"Loaded Step 2 checkpoint: {df.shape[0]:,} rows × {df.shape[1]} columns")

# 2. Sort chronologically by purchase timestamp so expanding window uses PAST orders only
df['order_purchase_timestamp'] = pd.to_datetime(df['order_purchase_timestamp'], errors='coerce')
df = df.sort_values('order_purchase_timestamp').reset_index(drop=True)

# Dynamic global baseline return rate
GLOBAL_RETURN_RATE = df['is_returned'].mean()
print(f"Global baseline return rate: {GLOBAL_RETURN_RATE:.4f}")

# =====================================================================
# CUSTOMER HISTORICAL FEATURES (Baseline = 0 / neutral for first-timers)
# =====================================================================
df['cum_customer_returns'] = df.groupby('customer_unique_id')['is_returned'].cumsum() - df['is_returned']
df['cum_customer_orders'] = df.groupby('customer_unique_id').cumcount()

df['customer_return_rate_fixed'] = np.where(
    df['cum_customer_orders'] > 0,
    df['cum_customer_returns'] / df['cum_customer_orders'],
    0.0
)
df['customer_order_count_fixed'] = df['cum_customer_orders']

df['customer_total_spend_fixed'] = (
    df.groupby('customer_unique_id')['price'].cumsum() - df['price']
)

df['cum_customer_reviews'] = df.groupby('customer_unique_id')['review_score'].cumsum() - df['review_score']
df['customer_avg_review_fixed'] = np.where(
    df['cum_customer_orders'] > 0,
    df['cum_customer_reviews'] / df['cum_customer_orders'],
    3.0
)

# Extreme reviewer detection via expanding standard deviation
cust_review_std = (
    df.groupby('customer_unique_id')['review_score']
      .expanding()
      .std()
      .reset_index(level=0, drop=True)
)

cust_review_std = (
    cust_review_std.groupby(df['customer_unique_id'])
                   .shift(1)
                   .fillna(0)
)
df['is_extreme_reviewer_fixed'] = (
    ((df['customer_avg_review_fixed'] <= 1.5) | (df['customer_avg_review_fixed'] >= 4.8)) &
    (df['customer_order_count_fixed'] >= 2) &
    (cust_review_std < 0.5)
).astype(int)

# =====================================================================
# CATEGORY & PRODUCT FEATURES (Baseline = Global Average)
# =====================================================================
df['cum_cat_returns'] = df.groupby('product_category_name_english')['is_returned'].cumsum() - df['is_returned']
df['cum_cat_orders'] = df.groupby('product_category_name_english').cumcount()
df['category_return_rate_fixed'] = np.where(
    df['cum_cat_orders'] > 0,
    df['cum_cat_returns'] / df['cum_cat_orders'],
    GLOBAL_RETURN_RATE
)

df['cum_product_returns'] = df.groupby('product_id')['is_returned'].cumsum() - df['is_returned']
df['cum_product_orders'] = df.groupby('product_id').cumcount()
df['product_return_rate_fixed'] = np.where(
    df['cum_product_orders'] > 0,
    df['cum_product_returns'] / df['cum_product_orders'],
    GLOBAL_RETURN_RATE
)
df['product_total_sales_fixed'] = df['cum_product_orders']

# =====================================================================
# CLEANUP & RENAME COLUMNS
# =====================================================================
temp_cols = ['cum_customer_returns', 'cum_customer_orders', 'cum_customer_reviews',
             'cum_product_returns', 'cum_product_orders', 'cum_cat_returns', 'cum_cat_orders']
df.drop(columns=temp_cols, inplace=True)

# Safely ignore old leaky columns if missing, and assign standardized names
leaky_cols = ['customer_return_rate', 'product_return_rate', 'category_return_rate',
              'customer_avg_review', 'customer_order_count', 'customer_total_spend',
              'is_extreme_reviewer', 'product_total_sales']
df.drop(columns=leaky_cols, errors='ignore', inplace=True)

df = df.rename(columns={
    'customer_return_rate_fixed': 'customer_return_rate',
    'product_return_rate_fixed': 'product_return_rate',
    'category_return_rate_fixed': 'category_return_rate',
    'customer_avg_review_fixed': 'customer_avg_review',
    'customer_order_count_fixed': 'customer_order_count',
    'customer_total_spend_fixed': 'customer_total_spend',
    'is_extreme_reviewer_fixed': 'is_extreme_reviewer',
    'product_total_sales_fixed': 'product_total_sales'
})

# Save output checkpoint for Step 4 (Haversine & Advanced Features)
df.to_csv('processed_return_data_no_leakage.csv', index=False)

print("=" * 60)
print("✅ LEAKAGE-FREE DATASET SAVED TO 'processed_return_data_no_leakage.csv'")
print("=" * 60)
print(f"Final shape: {df.shape[0]:,} rows × {df.shape[1]} columns\n")

Step 3: Calculating clean, leak-free historical features chronologically...
Loaded Step 2 checkpoint: 110,739 rows × 38 columns
Global baseline return rate: 0.1562
✅ LEAKAGE-FREE DATASET SAVED TO 'processed_return_data_no_leakage.csv'
Final shape: 110,739 rows × 46 columns

